# 06 - Sensitivity and reporting

This notebook closes the loop: estimate, stress-test, and write a reproducible causal report.


## Causal setup

- Question: what is the average effect of treatment on outcome?
- Treatment: `treatment`.
- Outcome: `outcome`.
- Unit: each synthetic individual.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.estimators import aipw_ate
from causal_inference_lab.reporting import CausalReport
from causal_inference_lab.sensitivity import omitted_confounder_simulation, placebo_treatment_test


In [ ]:
dataset = make_confounded_binary_treatment(n=5_000, seed=42)
data = dataset.data
covariates = ["x1", "x2", "x3"]

base = aipw_ate(data, covariates)
print(f"AIPW estimate: {base.estimate:.3f}")
print(f"True ATE (synthetic): {dataset.true_ate:.3f}")


## Step execution

The next cells perform uncertainty and robustness checks before reporting.


In [ ]:
placebo = placebo_treatment_test(
    data=data,
    estimator=aipw_ate,
    covariates=covariates,
    treatment_col="treatment",
    seed=123,
)
print(f"Reference estimate: {placebo.reference_estimate:.3f}")
print(f"Placebo estimate:   {placebo.estimate:.3f}")

sensitivity = omitted_confounder_simulation(
    data=data,
    base_effect=base.estimate,
    confounder_strength_grid=[0.0, 0.25, 0.5, 0.75],
    treatment_col="treatment",
)
print(sensitivity)

worst_bias = float(sensitivity.loc[sensitivity["confounder_strength"] == 0.75, "simulated_bias"].iloc[0])
best_adjusted = float(sensitivity.loc[sensitivity["confounder_strength"] == 0.0, "adjusted_effect"].iloc[0])
worst_adjusted = float(sensitivity.loc[sensitivity["confounder_strength"] == 0.75, "adjusted_effect"].iloc[0])
print(f"Worst omitted-confounder bias (+0.75): {worst_bias:+.3f}")
print(f"Adjusted effect range: [{worst_adjusted:.3f}, {best_adjusted:.3f}]")


## Causal report

The report object forces assumptions and limitations to be explicit alongside results.


In [ ]:
report = CausalReport(
    question="What is the causal effect of treatment on outcome?",
    estimand="ATE",
    identification_assumptions=(
        "Consistency",
        "No interference",
        "Conditional ignorability for treatment given x1, x2, x3",
        "Overlap in treatment probabilities",
    ),
    estimator="Augmented inverse probability weighting",
    diagnostics={
        "abs_placebo_estimate": float(abs(placebo.estimate)),
        "treated_minus_control_signal": float(abs(placebo.reference_estimate)),
        "worst_adjusted_effect_shift": float(best_adjusted - worst_adjusted),
    },
    uncertainty={
        "placebo_estimate": float(placebo.estimate),
        "worst_simulated_bias": float(worst_bias),
    },
    results={"AIPW": float(base.estimate), "true_ate": float(dataset.true_ate)},
    limitations=(
        "Unmeasured confounding remains a risk in non-random data.",
        "Functional form assumptions of outcome and treatment models may be misspecified.",
        "Results are benchmark-specific and not a production policy recommendation.",
    ),
    recommendation="Do not operationalize without out-of-sample validation and fairness review.",
)
print(report.to_markdown())
